# Exporting NNP models for GROMACS
In this example, we want to go over how to export models trained in Pytorch for use with the NNP interface in GROMACS. 

To learn how to wrap models so that they're compatible with the interface, look at some of the examples in the `models` folder. In general, they only need to define a forward function conforming to the following requirements:
1. The inputs should consist only of tensors representing one or several of the options given by the interface (e.g., atom positions, atomic numbers, simulation box vectors, PBCs, etc.). Remember that in GROMACS, inputs are passed to the model in order of their occurence in the `.mdp` file, so the order should match. 
2. The model should return a tensor containing the total energy of the NNP system. By default, GROMACS will then calculate the forces as negative gradients of the energy tensor w.r.t the _first_ input, which should be the positions. Optionally, these can be overriden by returning an additional force tensor, which is useful if the forces have to be calculated in a non-standard way.

### Checking the Pytorch version
To run simulations with the NNPot interface, it is important that GROMACS is compiled with the right version of Pytorch's C++ API, Libtorch, and CUDA. To find out what version you're using, run the following code:

In [2]:
import torch
print("Torch version:\t", torch.__version__)
print("CUDA available:\t", torch.cuda.is_available())
print("CUDA version:\t", torch.version.cuda)

Torch version:	 2.4.1.post302
CUDA available:	 True
CUDA version:	 12.0


## ANI models
First, let's export some pre-trained models from the [TorchANI package](https://github.com/aiqm/torchani). The following code is intended to work with the old version of TorchANI (pre-2.6). When you first use these models, the package will automatically download the pre-trained model weights from the Github repository. We'll use the ANI2x version, with the full ensemble. If you only want to use a single model, you can specify a model index from 0 to 7. This will speed up the calculation, but lead to less accurate results. 

In [ ]:
from models.gmx_ani import GmxANIModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'

save_path = 'models/ani2x.pt'
model = GmxANIModel(version=2, device=device)
torch.jit.script(model).save(save_path)

### NNPOps
To use the highly optimized NNPOps package to accelerate the inference even further, we have to tell the Libtorch version used by GROMACS about the extension. You can follow the instructions in the [GROMACS install guide](https://manual.gromacs.org/2025.0/install-guide/index.html#building-with-neural-network-potential-support), or alternatively, we can export the path to the extension with the model itself, so that the extension can be loaded at runtime. Another special feature is that NNPOps optimizations rely on static tensor shapes, which is why the exact form of the model needs to be known at compile-time. Therefore, we need to specify the atomic numbers used in the model. To match the sequence passed to the model by GROMACS, check the `.gro` file of the simulation.

In [ ]:
from models.gmx_ani import GmxANIModel
save_path = 'models/ani2x_nnpops.pt'

# example atomic number tensor for alanine dipeptide
atomic_numbers = torch.tensor([1,6,1,1,6,8,7,1,6,1,6,1,1,1,6,8,7,1,6,1,1,1], device=device)
model = GmxANIModel(use_opt='nnpops', atomic_numbers=atomic_numbers, version=2, device=device)

# nnpops can be found by checking for torch extension library
ext_lib = []
for lib in torch.ops.loaded_libraries:
    if lib:
        ext_lib.append(lib)
# if multiple extensions are found, they are separated by ':'
ext_lib = ":".join(ext_lib)
print("loaded extension libraries: ", ext_lib)
extra_files = {}
extra_files['extension_libs'] = ext_lib

torch.jit.script(model).save(save_path, _extra_files=extra_files)

### TorchANI 2.0
For the ANI family of models from TorchANI 2.0, the procedure is very much the same:

In [ ]:
from models.gmx_ani import GmxANIv2Model

save_path = 'models/ani2x.pt'
model = GmxANIv2Model()
torch.jit.script(model).save(save_path)

## EMLE
The [EMLE models](https://github.com/chemle/emle-engine) are designed to wrap themselves around an arbitrary NNP predicting *in vacuo* quantities, and provide an electrostatic embedding in hybrid ML/MM simulations. I.e., they include polarization effects of the surrounding MM region on the ML atoms. The EMLE package itself provides ready-made EMLE models wrapping ANI2x, as well as MACE.

In [ ]:
from models.gmx_emle import GmxEMLEModel

save_path = 'models/emle_ani2x_ala2.pt'
# decide if we want to use MACE or ANI. For MACE, the MACE-OFF23-small model is used by default.
flavor = 'ani2x'
# The EMLE model can also use NNPOps automatically, if atomic numbers of the ML region are provided
atomic_numbers = torch.tensor([1,6,1,1,6,8,7,1,6,1,6,1,1,1,6,8,7,1,6,1,1,1]) # alanine dipeptide
model = GmxEMLEModel(flavor, atomic_numbers=atomic_numbers)
torch.jit.script(model).save(save_path)

If we're using a model that handles electrostatic interactions with the MM region by itself (like EMLE), we also need to tell GROMACS about this (supported in version 2026.0 or newer). In this case, the interface needs to 
1. exclude ML-MM interactions from the long-range electrostatics calculation. Specifically, this is done by simply zeroing the charges. Note that LJ-interactions are still calculated.
2. provide the partial charges of the MM region as an input to the model
3. expect an output tensor containing the forces on the MM atoms.
To do this, we simply a line to the mdp file:
```
nnpot-embedding = electrostatic-model
```